<a href="https://colab.research.google.com/github/ileniadigital/Goodreads_analysis/blob/main/Book_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [95]:
# Import polars and pandas
import pandas as pd
import polars as pl
import matplotlib

In [96]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
PATH = '/content/drive/MyDrive/Colab Notebooks/'
print("Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [97]:
# Import data
pd_data = pd.read_csv(f'{PATH}goodreads_library_export.csv')
print("Pandas DataFrame:\n", pd_data.head())
pl_data = pl.read_csv(f'{PATH}goodreads_library_export.csv')
print("Polars DataFrame:\n", pl_data.head())

Pandas DataFrame:
     Book Id                                              Title  \
0  26008840  Les Disparus du Clairdelune (La Passe-Miroir, #2)   
1  40969531        A Winter's Promise (The Mirror Visitor, #1)   
2    117833                           The Master and Margarita   
3      3836                                        Don Quixote   
4     19089                                        Middlemarch   

                         Author                     Author l-f  \
0              Christelle Dabos              Dabos, Christelle   
1              Christelle Dabos              Dabos, Christelle   
2              Mikhail Bulgakov              Bulgakov, Mikhail   
3  Miguel de Cervantes Saavedra  Saavedra, Miguel de Cervantes   
4                  George Eliot                  Eliot, George   

                                  Additional Authors           ISBN  \
0                                                NaN  ="2070661989"   
1                                   Hildegard

In [98]:
# Data columns
# pd_data.columns
print(type(pd_data))

<class 'pandas.core.frame.DataFrame'>


In [99]:
# DataFrame cleanup
pd_data = pd_data.drop(columns=['ISBN', 'ISBN13','Binding', 'Spoiler',
                      'Private Notes', 'Owned Copies'])
# Rename columns to remove space
pd_data.columns = pd_data.columns.str.lower().str.replace(' ', '_')
print("---Book(s) I am currently reading---\n")
current = pd_data[
    (pd_data['read_count'] == 1) &
    (pd_data['exclusive_shelf'] == 'currently-reading')
]
print(current.head())

print("\n---Extract books already read---")
past = pd_data[
    (pd_data['read_count'] == 1) &
    (pd_data['exclusive_shelf'] == 'read')
]
print(past.head())

# Calculate statistics of past read books
print(f'Average rating for read books: {(past['my_rating'].mean())}')
print(f'Standard deviation of rating for read books: {(past['my_rating'].std())}')
print(f'Smallest rating for read books: {(past['my_rating'].min())}')
print(f'Greatest rating for read books: {(past['my_rating'].max())}')

---Book(s) I am currently reading---

    book_id                                              title  \
0  26008840  Les Disparus du Clairdelune (La Passe-Miroir, #2)   

             author         author_l-f additional_authors  my_rating  \
0  Christelle Dabos  Dabos, Christelle                NaN        0.0   

   publisher  number_of_pages  year_published  original_publication_year  \
0  Gallimard            560.0          2015.0                     2015.0   

  date_read  date_added        bookshelves bookshelves_with_positions  \
0       NaN  2026/08/15  currently-reading     currently-reading (#1)   

     exclusive_shelf my_review  read_count  
0  currently-reading       NaN           1  

---Extract books already read---
    book_id                                        title            author  \
1  40969531  A Winter's Promise (The Mirror Visitor, #1)  Christelle Dabos   
6    199264     The Temple: The Poetry of George Herbert    George Herbert   
7     10775               

In [114]:
# Analysis using polars
pl_data = pl.DataFrame(pl_data)

# Delete unnecessary columns
# pl_data = pl_data.drop(['ISBN', 'ISBN13','Binding', 'Spoiler',
#                       'Private Notes', 'Owned Copies'])
# Rename columns
pl_data = pl_data.rename(lambda column_name: column_name.lower().replace(' ', '_'))
# Extract currently reading book
current = pl_data.filter(
    (pl.col('exclusive_shelf') == 'currently-reading')
)
print(current.head())
# Extract read books
past = pl_data.filter(
    (pl.col('my_rating') == 1) &
    (pl.col('exclusive_shelf') == 'read')
)
print(past.head())
# Calculate statistics about read books

shape: (1, 23)
┌──────────┬────────────┬────────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
│ book_id  ┆ title      ┆ author     ┆ author_l- ┆ … ┆ spoiler ┆ private_n ┆ read_coun ┆ owned_cop │
│ ---      ┆ ---        ┆ ---        ┆ f         ┆   ┆ ---     ┆ otes      ┆ t         ┆ ies       │
│ i64      ┆ str        ┆ str        ┆ ---       ┆   ┆ str     ┆ ---       ┆ ---       ┆ ---       │
│          ┆            ┆            ┆ str       ┆   ┆         ┆ str       ┆ i64       ┆ i64       │
╞══════════╪════════════╪════════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
│ 26008840 ┆ Les        ┆ Christelle ┆ Dabos,    ┆ … ┆ null    ┆ null      ┆ 1         ┆ 0         │
│          ┆ Disparus   ┆ Dabos      ┆ Christell ┆   ┆         ┆           ┆           ┆           │
│          ┆ du Clairde ┆            ┆ e         ┆   ┆         ┆           ┆           ┆           │
│          ┆ lune (L…   ┆            ┆           ┆   ┆         ┆           ┆